# 对冲实验：模型误设与交易成本（Step 6）

真实路径：Heston；对冲：BS Delta（波动率可能误设）。另扫描比例交易成本与再平衡频率 N 的权衡。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from hedging import misspec_study, plot_cost_tradeoff, plot_misspec_pnl
from processes import default_misspecified_heston_params

FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
S0, K, T, r = 100.0, 100.0, 1.0, 0.05

In [ ]:
params = default_misspecified_heston_params()
stats = misspec_study(S0=S0, K=K, T=T, r=r, params=params, n_paths=20_000, seed=42)
for st in stats:
    print(
        st.label,
        f"mean={st.mean:.3f}",
        f"std={st.std:.3f}",
        f"skew={st.skewness:.3f}",
        f"kurt={st.excess_kurtosis:.3f}",
        f"q05={st.q05:.3f}",
        f"q01={st.q01:.3f}",
    )

plot_misspec_pnl(
    S0=S0, K=K, T=T, r=r, rho=-0.7,
    out_path=str(FIG / "fig_misspec_pnl.png"),
)

In [ ]:
# 假设比例费率 k（非真实市场费率）：0.05% 与 0.1%
opt = plot_cost_tradeoff(
    S0=S0, K=K, T=T, r=r, sigma=0.2,
    cost_ks=(0.0, 0.0005, 0.001),
    out_path=str(FIG / "fig_cost_tradeoff.png"),
)
print("启发式最优 N（左尾+波动）:", opt)